# 🧠 Neural Networks — Code Demos

**Python ML Course — Week 23 | Learn and Help**

This notebook contains all the code from the [Neural Networks Presentation](https://github.com/sjasthi/Python-ML-Machine-Learning/blob/main/Presentations/ML_Neural_Networks.md).

We solve the **same problem** — classifying handwritten digits (0–9) from the **MNIST dataset** (70,000 images) — using three different tools:

| Approach | Tool | Difficulty |
|----------|------|-----------|
| 1 | scikit-learn (MLPClassifier) | ⭐ Easy |
| 2 | Keras (Sequential) | ⭐⭐ Medium |
| 3 | TensorFlow (Low-Level) | ⭐⭐⭐ Advanced |

Run each section and compare the results!

---

## Setup

Install required packages (run once):

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install scikit-learn tensorflow numpy

---

## Approach 1: scikit-learn (MLPClassifier) — The Familiar Way 🟢

You already know scikit-learn! It has a built-in neural network called `MLPClassifier` (Multi-Layer Perceptron). Same `.fit()` and `.predict()` you've used all year.

**What's familiar:** `fit()`, `predict()`, `accuracy_score()` — exactly like Random Forest or KNN!

**What's new:** `MLPClassifier` with `hidden_layer_sizes` to define the network shape.

In [ ]:
# ===========================================
# Neural Network with scikit-learn
# ===========================================
# Uses the SAME API you already know!

from sklearn.neural_network import MLPClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import time

# Step 1: Load the MNIST dataset
print("Loading MNIST dataset...")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

# Step 2: Prepare the data
X = X / 255.0  # Scale pixel values from 0-255 to 0.0-1.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Build and train the Neural Network
# hidden_layer_sizes=(128, 64) means:
#   - Hidden Layer 1: 128 neurons
#   - Hidden Layer 2: 64 neurons
print("\n🧠 Training Neural Network (scikit-learn)...")
print("=" * 50)

start = time.time()
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),   # Two hidden layers
    activation='relu',               # Activation function (dimmer switch!)
    max_iter=20,                     # Maximum training epochs
    random_state=42,
    verbose=True                     # Show progress while training
)
mlp.fit(X_train, y_train)
train_time = time.time() - start

# Step 4: Test the model
y_pred = mlp.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n{'=' * 50}")
print(f"🎯 Test Accuracy: {accuracy * 100:.2f}%")
print(f"⏱️  Training Time: {train_time:.1f} seconds")
print(f"\nFirst 10 predictions vs actual:")
for i in range(10):
    status = "✅" if y_pred[i] == y_test[i] else "❌"
    print(f"  {status} Predicted: {y_pred[i]} | Actual: {y_test[i]}")

---

## Approach 2: Keras — The Beginner-Friendly Deep Learning Way 🟡

Keras is designed to make deep learning simple. It's the most popular way to build neural networks in industry.

**What's different from scikit-learn:**
- You build the model layer-by-layer with `Sequential()`
- You must `compile()` before training (choose optimizer + loss function)
- You get **confidence scores** for each prediction (e.g., "92% sure it's a 7")
- Training shows a progress bar with live accuracy updates

In [ ]:
# ===========================================
# Neural Network with Keras
# ===========================================
# Keras makes deep learning feel simple!

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import time

# Step 1: Load the MNIST dataset (Keras has it built-in!)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Step 2: Prepare the data
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

# Step 3: Build the Neural Network layer by layer
model = keras.Sequential([
    layers.Dense(128, activation="relu", input_shape=(784,)),  # Hidden Layer 1
    layers.Dense(64, activation="relu"),                        # Hidden Layer 2
    layers.Dense(10, activation="softmax")                      # Output Layer (10 digits)
])

# Step 4: Compile — tell Keras HOW to train
model.compile(
    optimizer="adam",                          # Weight adjustment algorithm
    loss="sparse_categorical_crossentropy",    # Error measurement for classification
    metrics=["accuracy"]
)

# Step 5: Train!
print("🧠 Training Neural Network (Keras)...")
print("=" * 50)
start = time.time()
model.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.1)
train_time = time.time() - start

# Step 6: Test
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"\n{'=' * 50}")
print(f"🎯 Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"⏱️  Training Time: {train_time:.1f} seconds")

# Step 7: Predictions with confidence scores
predictions = model.predict(x_test[:10])
print(f"\nFirst 10 predictions:")
for i in range(10):
    predicted = np.argmax(predictions[i])
    confidence = predictions[i][predicted] * 100
    actual = y_test[i]
    status = "✅" if predicted == actual else "❌"
    print(f"  {status} Predicted: {predicted} | Actual: {actual} | Confidence: {confidence:.1f}%")

---

## Approach 3: TensorFlow (Low-Level) — The "Under the Hood" Way 🔴

TensorFlow gives you full control. This is what researchers and engineers use when they need maximum flexibility. It's more code, but you see every step.

**What's different from Keras:**
- YOU define the weights and biases as variables
- YOU write the forward pass function (`neural_network()`)
- YOU write the training loop (Keras does this behind the scenes)
- You can see the **exact math** happening: `matmul` (matrix multiply), `relu`, `softmax`

In [ ]:
# ===========================================
# Neural Network with TensorFlow (Low-Level)
# ===========================================
# More control — you see every step!

import numpy as np
import tensorflow as tf
import time

# Step 1: Load and prepare data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

# Convert labels to one-hot encoding
# Instead of label "3", we use [0,0,0,1,0,0,0,0,0,0]
y_train_onehot = tf.one_hot(y_train, depth=10)
y_test_onehot = tf.one_hot(y_test, depth=10)

# Step 2: Create the dataset pipeline
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train_onehot))
train_dataset = train_dataset.shuffle(10000).batch(32)

# Step 3: Define network parameters (weights and biases)
# These are the numbers the network will LEARN!
W1 = tf.Variable(tf.random.normal([784, 128], stddev=0.1))  # Weights: Input -> Hidden 1
b1 = tf.Variable(tf.zeros([128]))                            # Biases: Hidden 1

W2 = tf.Variable(tf.random.normal([128, 64], stddev=0.1))   # Weights: Hidden 1 -> Hidden 2
b2 = tf.Variable(tf.zeros([64]))                             # Biases: Hidden 2

W3 = tf.Variable(tf.random.normal([64, 10], stddev=0.1))    # Weights: Hidden 2 -> Output
b3 = tf.Variable(tf.zeros([10]))                             # Biases: Output

# Step 4: Define how data flows through the network (forward pass)
def neural_network(x):
    # Layer 1: multiply inputs by weights, add bias, apply ReLU
    layer1 = tf.nn.relu(tf.matmul(x, W1) + b1)
    # Layer 2: same thing
    layer2 = tf.nn.relu(tf.matmul(layer1, W2) + b2)
    # Output: use softmax to get probabilities
    output = tf.nn.softmax(tf.matmul(layer2, W3) + b3)
    return output

# Step 5: Define the optimizer (how to adjust weights)
optimizer = tf.optimizers.Adam(learning_rate=0.001)

# Step 6: Training loop — this is what Keras hides from you!
print("🧠 Training Neural Network (TensorFlow)...")
print("=" * 50)
start = time.time()

for epoch in range(5):
    epoch_loss = 0
    num_batches = 0

    for batch_x, batch_y in train_dataset:
        with tf.GradientTape() as tape:
            # Forward pass: make predictions
            predictions = neural_network(batch_x)
            # Calculate loss (how wrong are we?)
            loss = -tf.reduce_mean(tf.reduce_sum(batch_y * tf.math.log(predictions + 1e-7), axis=1))

        # Backward pass: calculate gradients and update weights
        gradients = tape.gradient(loss, [W1, b1, W2, b2, W3, b3])
        optimizer.apply_gradients(zip(gradients, [W1, b1, W2, b2, W3, b3]))

        epoch_loss += loss.numpy()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    print(f"  Epoch {epoch+1}/5 — Loss: {avg_loss:.4f}")

train_time = time.time() - start

# Step 7: Test
test_predictions = neural_network(x_test)
predicted_labels = tf.argmax(test_predictions, axis=1).numpy()
accuracy = np.mean(predicted_labels == y_test)

print(f"\n{'=' * 50}")
print(f"🎯 Test Accuracy: {accuracy * 100:.2f}%")
print(f"⏱️  Training Time: {train_time:.1f} seconds")

print(f"\nFirst 10 predictions:")
for i in range(10):
    pred = predicted_labels[i]
    conf = test_predictions[i][pred].numpy() * 100
    actual = y_test[i]
    status = "✅" if pred == actual else "❌"
    print(f"  {status} Predicted: {pred} | Actual: {actual} | Confidence: {conf:.1f}%")

---

## 🏆 Comparing All Three Approaches

| Feature | scikit-learn | Keras | TensorFlow |
|---------|-------------|-------|------------|
| **Lines of code** | ~15 | ~25 | ~50 |
| **Difficulty** | ⭐ Easy | ⭐⭐ Medium | ⭐⭐⭐ Advanced |
| **API style** | `fit()` / `predict()` | Layer-by-layer + `compile()` | Write everything yourself |
| **Best for** | Quick experiments, learning | Most real projects | Custom research, max control |
| **GPU support** | ❌ No | ✅ Yes | ✅ Yes |
| **CNNs, RNNs, Transformers** | ❌ No | ✅ Yes | ✅ Yes |
| **Expected accuracy** | ~97% | ~97-98% | ~97-98% |

> 💡 **Key Takeaway:** All three solve the same problem with similar accuracy! The difference is how much control (and code) you want.
>
> **scikit-learn = automatic car** | **Keras = manual car** | **TensorFlow = building the engine yourself**

---

## 🏁 Ultimate Comparison Script

Run all three approaches back-to-back and compare results in a final scoreboard!

In [ ]:
# ===========================================
# ULTIMATE COMPARISON: scikit-learn vs Keras vs TensorFlow
# ===========================================
# Same problem. Same data. Three tools. Who wins?

import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
from tensorflow.keras import layers
import time

# ---- Load Data ----
print("📦 Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train_flat = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test_flat = x_test.reshape(-1, 784).astype("float32") / 255.0

results = {}

# ---- 1. SCIKIT-LEARN ----
print("\n" + "=" * 60)
print("🟢 APPROACH 1: scikit-learn MLPClassifier")
print("=" * 60)
start = time.time()
mlp = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu',
                    max_iter=20, random_state=42, verbose=False)
mlp.fit(x_train_flat, y_train)
sklearn_time = time.time() - start
sklearn_acc = accuracy_score(y_test, mlp.predict(x_test_flat))
results['scikit-learn'] = (sklearn_acc, sklearn_time)
print(f"   🎯 Accuracy: {sklearn_acc * 100:.2f}%")
print(f"   ⏱️  Time: {sklearn_time:.1f}s")

# ---- 2. KERAS ----
print("\n" + "=" * 60)
print("🟡 APPROACH 2: Keras Sequential")
print("=" * 60)
model = keras.Sequential([
    layers.Dense(128, activation="relu", input_shape=(784,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
start = time.time()
model.fit(x_train_flat, y_train, epochs=5, batch_size=32, verbose=0)
keras_time = time.time() - start
keras_loss, keras_acc = model.evaluate(x_test_flat, y_test, verbose=0)
results['Keras'] = (keras_acc, keras_time)
print(f"   🎯 Accuracy: {keras_acc * 100:.2f}%")
print(f"   ⏱️  Time: {keras_time:.1f}s")

# ---- 3. SCIKIT-LEARN (Random Forest — for reference) ----
print("\n" + "=" * 60)
print("🌲 BONUS: Random Forest (Classic ML — not a neural network)")
print("=" * 60)
from sklearn.ensemble import RandomForestClassifier
start = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train_flat[:10000], y_train[:10000])  # Smaller subset
rf_time = time.time() - start
rf_acc = accuracy_score(y_test, rf.predict(x_test_flat))
results['Random Forest'] = (rf_acc, rf_time)
print(f"   🎯 Accuracy: {rf_acc * 100:.2f}%  (trained on 10K samples)")
print(f"   ⏱️  Time: {rf_time:.1f}s")

# ---- FINAL SCOREBOARD ----
print("\n" + "=" * 60)
print("🏆 FINAL SCOREBOARD")
print("=" * 60)
print(f"{'Approach':<20} {'Accuracy':<12} {'Time':<10} {'Type'}")
print("-" * 60)
for name, (acc, t) in results.items():
    ntype = "Neural Net" if name != "Random Forest" else "Classic ML"
    print(f"{name:<20} {acc*100:.2f}%       {t:.1f}s       {ntype}")
print("-" * 60)
winner = max(results, key=lambda k: results[k][0])
print(f"\n🥇 Highest Accuracy: {winner}")
print(f"\n💡 Notice: All approaches get similar accuracy on this task!")
print(f"   Neural networks shine more on complex tasks like full images, text, and audio.")

---

## 📝 Record Your Results

Fill in after running the Ultimate Comparison Script:

| Approach | Accuracy | Training Time | Type |
|----------|----------|---------------|------|
| scikit-learn (MLP) | ____% | ____s | Neural Network |
| Keras | ____% | ____s | Neural Network |
| Random Forest | ____% | ____s | Classic ML |

---

*Source: [ML_Neural_Networks.md](https://github.com/sjasthi/Python-ML-Machine-Learning/blob/main/Presentations/ML_Neural_Networks.md)*